# 20 · Multi-Agent — Sequential Pipeline

A **Supervisor** (notebook 16) routes a task to *one* worker. A **Pipeline**
instead chains workers in a fixed order, each consuming the previous one's
output — the assembly line: **Planner → Research → Coder → Reviewer → Tester**.
Each agent has a single narrow responsibility.

> Runs on Ollama. Swap `provider=` on any `Agent` to use another model.

In [ ]:
import sys, pathlib
p = pathlib.Path.cwd()
root = next(x for x in [p, *p.parents] if (x / 'agents_lab').exists())
sys.path.insert(0, str(root))
import agents_lab  # noqa: F401  (bootstraps sibling study_buddy + rag_lab)

Build five specialists, each with a role in its system prompt, then wire them
into a `Pipeline`.

In [ ]:
from agents_lab import Agent, Worker, Pipeline

roles = {
    'Planner':  'Break the request into a short, ordered plan. Output only the plan.',
    'Research': 'Given the plan, list the key facts, APIs, or edge cases to consider.',
    'Coder':    'Write the code that implements the plan using the research. Output code only.',
    'Reviewer': 'Critique the code for bugs and clarity; list concrete fixes.',
    'Tester':   'Given the code and review, write test cases and the final verdict.',
}
stages = [Worker(name, desc, Agent(provider='ollama', system=desc)) for name, desc in roles.items()]
pipe = Pipeline(stages)

In [ ]:
result = pipe.run('Write a Python function that returns the nth Fibonacci number.')
for name, output in result.outputs:
    print(f'\n===== {name} =====\n{output[:400]}')
print('\n\n##### FINAL (Tester) #####\n', result.final[:500])

**Takeaway:** a pipeline turns one hard task into a chain of easy, auditable
hand-offs — the shape of your Project 5. Each stage stays simple because it
only sees the prior output. Deep orchestration (branching, retries,
checkpoints) is Phase 5.

Next → `21_group_chat.ipynb`